# LLM Topic Labeling

Uses an LLM to replace auto-generated BERTopic keyword labels with concise, human-readable topic names and descriptions.

## Prerequisites

The following must be available **in memory** (run `10-topic_modelling.ipynb` first):

| Variable | Source | Description |
|---|---|---|
| `topic_model` | `10-topic_modelling.ipynb` (fit or loaded from `./BERTopic/`) | Fitted BERTopic model |
| `docs` | `10-topic_modelling.ipynb` | List of document strings used to fit the model |

**Output produced by this notebook:**

| File | Description |
|---|---|
| `./pickles/topic_metadata_llm.pkl` | DataFrame with `Topic`, `CustomLabel`, `Description` per topic |

In [16]:
import pandas as pd
from bertopic import BERTopic

# ── Load BERTopic model ──────────────────────────────────────────────────────
# Saved by 10-topic_modelling.ipynb via:
#   topic_model.save("./BERTopic", serialization="safetensors", save_ctfidf=True,
#                    save_embedding_model="all-MiniLM-L6-v2")
topic_model = BERTopic.load("./BERTopic", embedding_model="all-MiniLM-L6-v2")

# ── Reconstruct docs ─────────────────────────────────────────────────────────
# Must match the exact construction used when the model was fitted.
df = pd.read_pickle("./pickles/first-date_posts-all.pkl")
documents = df["title"] + ". " + df["selftext"].fillna("")
documents = documents.apply(lambda x: x.replace("\n", " "))
docs = documents.tolist()

print(f"✅ Loaded topic model with {len(topic_model.get_topic_info())} topics.")
print(f"✅ Reconstructed {len(docs)} documents.")

✅ Loaded topic model with 40 topics.
✅ Reconstructed 87155 documents.


## 1. LLM Topic Labeling

For each BERTopic topic, query an LLM (Gemma 3 27B via Ollama) with the top-N keywords and 5 most representative documents to generate a human-readable label and short description.

Iterates all topics (skipping outlier `-1`), collects results, applies labels to the in-memory model via `set_topic_labels`, and saves the full metadata to a pickle.

**To adjust:** Set `local_requests = True` to use a local Ollama instance instead of the remote API. Change the model name (`gemma3:27b`) or the number of representative docs to tune quality vs. speed.

In [17]:
import os
import re

from dotenv import load_dotenv
from ollama import chat
from ollama import ChatResponse
from ollama import Client

load_dotenv()

# ── Configuration ─────────────────────────────────────────────────────────────
local_requests = False
OLLAMA_API_KEY = os.environ["OLLAMA_API_KEY"]

# topic_model.topics_ is a list of topic assignments per document — always persisted.
# Zip once here so we don't repeat it inside every function call.
docs_by_topic = {}
for doc, t in zip(docs, topic_model.topics_):
    docs_by_topic.setdefault(t, []).append(doc)


def get_topic_label_and_description(topic_id):
    topic_info = topic_model.get_topic_info(topic_id)

    representative_docs = docs_by_topic.get(topic_id, [])[:5]
    full_formatted_docs_string = "\n".join(representative_docs)

    top_words = ', '.join(topic_info['Representation'].iloc[0])

    prompt = f"""
    You are a dating culture and dating scripts specialist.

    Your task: Based on top n words, and some representative documents, create a logical topic label and a small description of a topic (max three sentences).
    Be clear and precise as possible.

    --INPUT BEGIN--
    Top n words: [{top_words}]
    Representative docs: [{full_formatted_docs_string[:10000]}]
    --INPUT END--

    Provide output exactly in this format:
    Topic label: {{label}}
    Description: {{description}}
    """

    if local_requests:
        response: ChatResponse = chat(model='gemma3', messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ])
        llm_response = response['message']['content']
    else:
        client = Client(
            host="https://ollama.com",
            headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'}
        )

        messages = [
            {
                'role': 'user',
                'content': prompt,
            },
        ]

        llm_response = ""
        for part in client.chat('gemma3:27b', messages=messages, stream=True):
            chunk = part.get('message', {}).get('content')
            if chunk:
                llm_response += chunk

    pattern = r"Topic label: (?P<label>.*?)\nDescription: (?P<desc>.*)"

    # re.DOTALL so the description is captured even if it spans multiple lines
    match = re.search(pattern, llm_response, re.DOTALL)

    topic_label = ""
    description = ""

    if match:
        topic_label = match.group('label').strip()
        description = match.group('desc').strip()

    return {"label": topic_label, "description": description}


topics = topic_model.get_topic_info()

topic_labels_map = {}
topic_metadata_list = []

print("--- Starting Topic Renaming and Description Collection ---")

for index, row in topics.iterrows():
    topic_id = row['Topic']

    if topic_id == -1:
        continue

    new_topic_info = get_topic_label_and_description(topic_id)
    new_label = new_topic_info["label"]
    new_description = new_topic_info["description"]

    if not new_label:
        new_label = row['Name']
        new_description = "No description generated."

    print(f"Topic {topic_id}: {new_label}")

    topic_labels_map[topic_id] = new_label
    topic_metadata_list.append({
        'Topic': topic_id,
        'CustomLabel': new_label,
        'Description': new_description
    })

# --- EXPORT ---
metadata_df = pd.DataFrame(topic_metadata_list)

output_path = './pickles/topic_metadata_llm.pkl'
metadata_df.to_pickle(output_path)
print(f"\n✅ Topic metadata (ID, Label, Description) saved to: {output_path}")

if topic_labels_map:
    topic_model.set_topic_labels(topic_labels_map)
    print("✅ Topic labels applied to current BERTopic model.")
else:
    print("No topic labels were updated.")

--- Starting Topic Renaming and Description Collection ---
Topic 0: First Date Navigation
Topic 1: Post-First Date Follow-Up & Communication
Topic 2: Modern Dating Frustrations & Self-Perception
Topic 3: Early Dating Texting Etiquette
Topic 4: Dating Payment Expectations
Topic 5: Dating Ghosting & Post-Date Disappearance
Topic 6: Sudden Ghosting After Initial Connection
Topic 7: Dating & Gift-Giving Etiquette (Flowers)
Topic 8: First Date Attire & Anxiety
Topic 9: Early Sexual Expectations in Dating
Topic 10: First/Second Date Kissing Dynamics
Topic 11: Relationship Betrayal & Post-Breakup Distress
Topic 12: Dating with Parental/Family Complications & Past Trauma
Topic 13: Early Dating & Sexual Expectations/Deception
Topic 14: Early Sexual Encounters & Shifting Dynamics
Topic 15: First Kiss Navigation
Topic 16: First Date Ghosting
Topic 17: Dating & Physical Attraction Standards
Topic 18: Tinder Dating Experiences & Red Flags
Topic 19: Early Stage Dating Uncertainty - Tinder
Topic 20: 